# 머신러닝응용 MBTI 성격 예측 모델 프로젝트
## 최종 코드

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV

# 데이터
df = pd.read_csv('https://raw.githubusercontent.com/tysep16/26_02_ML/refs/heads/main/16p.csv')
# 전처리
qtext = pd.Series(df.columns[1:61], index=[f'Q{i}' for i in range(1, 61)])
df.columns = ['id'] + list(qtext.index) + ['type']
x = df.drop(['id', 'type'], axis=1)
y = df['type']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

l1 = LogisticRegression(solver='saga', l1_ratio=1, C=0.01, max_iter=1000)
l1.fit(x_train, y_train) # Accuracy 0.918
lr42 = LogisticRegression(max_iter=1000)
lr42.fit(x_train.loc[:, l1.coef_.mean(axis=0) != 0], y_train) # Accuracy 0.913

knn = KNeighborsClassifier()
knn.fit(x_train, y_train) # Accuracy 0.989
knn.fit(x_train.loc[:, l1.coef_.mean(axis=0) != 0], y_train) # Accuracy 0.989

dt = DecisionTreeClassifier(random_state=42)
dt.fit(x_train, y_train)
dt1 = DecisionTreeClassifier(random_state=42)
dt1.fit(x_train.loc[:, l1.coef_.mean(axis=0) != 0], y_train)
print('원래 피쳐 : ', dt.score(x_test, y_test),
    '\t 나무 깊이 :', dt.get_depth(), '\t 리프 노드 수 :', dt.get_n_leaves())
print('줄인 피쳐 : ', dt1.score(x_test.loc[:, l1.coef_.mean(axis=0) != 0], y_test),
    '\t 나무 깊이 :', dt1.get_depth(), '\t 리프 노드 수 :', dt1.get_n_leaves())

pipe = Pipeline([
    ('select', SelectKBest(f_classif, k=40)), # 전처리 = 문항 40개 고르기
    ('tree', DecisionTreeClassifier(random_state=42)), # 모델 = 트리로 분류
])
pipe.fit(x_train, y_train)
chosen = x_train.columns[pipe.named_steps.select.get_support()]
print('고른 문항 수 :', len(chosen), '\n고른 문항 : ', chosen, '\n정확도 : ', pipe.score(x_test, y_test))

r = cross_validate(dt, x_train, y_train, cv=5, return_train_score=True)
r1 = cross_validate(pipe, x_train, y_train, cv=5, return_train_score=True)
print('dt: 교차검증 정확도 :', r['test_score'], '\t평가 정확도 : ', dt.score(x_test, y_test))
print('pipe: 교차검증 정확도 :', r1['test_score'], '\t평가 정확도 : ', pipe.score(x_test, y_test))

params = {
    'select__k': [20, 40, 60], # 문항 수
    'tree__max_depth': [5, 10, 15, None], # 질문 횟수(깊이)
    'tree__min_samples_leaf': [1, 5, 20], # 끝 노드 최소 인원
}
gs = GridSearchCV(pipe, params, cv=5, n_jobs=-1, return_train_score=True)
gs.fit(x_train, y_train)
print('최적 조합 :', gs.best_params_)
print('검증 점수 :', round(gs.best_score_, 3))

best = gs.best_estimator_
print('기본 트리 :', dt.score(x_test, y_test))
print('최적 트리 :', accuracy_score(y_test, gs.best_estimator_.predict(x_test)))

원래 피쳐 :  0.6488333333333334 	 나무 깊이 : 30 	 리프 노드 수 : 10680
줄인 피쳐 :  0.65825 	 나무 깊이 : 30 	 리프 노드 수 : 10732
고른 문항 수 : 40 
고른 문항 :  Index(['Q3', 'Q4', 'Q5', 'Q6', 'Q9', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16',
       'Q17', 'Q18', 'Q19', 'Q20', 'Q22', 'Q25', 'Q26', 'Q27', 'Q29', 'Q30',
       'Q31', 'Q33', 'Q35', 'Q36', 'Q37', 'Q38', 'Q39', 'Q40', 'Q41', 'Q43',
       'Q44', 'Q45', 'Q47', 'Q49', 'Q53', 'Q55', 'Q58', 'Q59', 'Q60'],
      dtype='str') 
정확도 :  0.6515
dt: 교차검증 정확도 : [0.63635417 0.63822917 0.63177083 0.64       0.6380873 ] 	평가 정확도 :  0.6488333333333334
pipe: 교차검증 정확도 : [0.63552083 0.63489583 0.6303125  0.65010417 0.63881654] 	평가 정확도 :  0.6515
최적 조합 : {'select__k': 40, 'tree__max_depth': 15, 'tree__min_samples_leaf': 1}
검증 점수 : 0.648
기본 트리 : 0.6488333333333334
최적 트리 : 0.6618333333333334


## 최종 결과

* Logistic
  * 60문항 → 0.918, 중요 문항만 → 0.913
  * 문항을 줄여도 정확도 거의 같음

* KNN
  * 60문항 → 0.989, 중요 문항만 → 0.989
  * 가장 정확한 모델

* Decision Tree
  * 60문항 → 0.649, 중요 문항만 → 0.658
  * 너무 복잡하게 학습해서 새로운 데이터는 잘 못 맞힘

* Pipeline, Cross Validation
  * 문항 40개만 골라 학습 → 0.652
  * 여러 번 나눠 시험해도 점수가 비슷 → 믿을 만한 결과

* Grid Search
  * 가장 좋은 설정 찾기 → 0.662

* 정리
  * KNN (0.989) > Logistic (0.918) > Decision Tree (0.662)
  * 60문항 대신 40문항만 물어봐도 MBTI를 비슷하게 맞힐 수 있음